# 1. Read source data

In [52]:
import sys
import numpy as np
import pandas as pd
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)
from config import PER_ORGANISM_PER_CLASS_ALLELES
from mhcgnomes import parse





In [53]:
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber/"
total_df =  pd.read_csv("{}schreiber_total_data.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))

per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_mt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    per_allele_wt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None

    
ALLELES

['H-2IEd', 'H-2Ld']

# 3. Read Joined predictions

In [54]:

#DATA_PATH_PREDICTIONS = "/home/denklewer/Data/gbm122/" #ubuntu machine
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber/"
# for decoy_key in DECOY_PREFIXES:
per_allele_predictions = dict()
all_methods = set()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df = pd.read_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";")
    per_allele_predictions[allele_name] = total_df
    all_methods.update(list(set([item.split('_')[0] for item in total_df.columns if  item.split('_')[0] not in ['allele', 'peptide']])))


In [55]:
total_df.columns

Index(['allele', 'peptide_mt', 'netMHC-3.4_score_mt', 'netMHC-3.4_affinity_mt',
       'netMHC-3.4_percentile_rank_mt', 'netMHC-4.0_score_mt',
       'netMHC-4.0_affinity_mt', 'netMHC-4.0_percentile_rank_mt',
       'netMHCpan-4.1_score_ba_mt', 'netMHCpan-4.1_score_el_mt',
       'netMHCpan-4.1_affinity_mt', 'netMHCpan-4.1_percentile_rank_ba_mt',
       'netMHCpan-4.1_percentile_rank_el_mt', 'mhcnuggets_score_mt',
       'mhcnuggets_affinity_mt', 'mhcnuggets_percentile_rank_mt',
       'puffin_score_mt', 'puffin_epistemic_var_mt', 'puffin_aleatoric_var_mt',
       'puffin_binding_likelihood_mt', 'puffin_percentile_rank_mt',
       'netMHC-3.4_score_advantage', 'netMHC-3.4_affinity_advantage',
       'netMHC-4.0_score_advantage', 'netMHC-4.0_affinity_advantage',
       'netMHC-4.0_percentile_rank_advantage',
       'netMHCpan-4.1_score_ba_advantage', 'netMHCpan-4.1_score_el_advantage',
       'netMHCpan-4.1_affinity_advantage',
       'netMHCpan-4.1_percentile_rank_ba_advantage',
      

In [56]:
all_methods

{'mhcnuggets',
 'netMHC-3.4',
 'netMHC-4.0',
 'netMHCIIpan-4.0',
 'netMHCpan-4.1',
 'puffin'}

# 4. Apply filtering

In [45]:
def filter_epitopes(df, method_name, advantage = False):
    if method_name == 'netMHC-3.4':
        df = df.loc[df[method_name + "_affinity_mt"] < 500].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif  method_name == 'netMHC-4.0':
        df = df.loc[df[method_name +  "_percentile_rank_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
    elif method_name == 'netMHCpan-4.1':
        df = df.loc[df[method_name + "_percentile_rank_el_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_el_advantage"] > 1].copy()
    elif method_name == 'netMHCIIpan-4.0':
        df = df.loc[df[method_name + "_percentile_rank_el_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_el_advantage"] > 1].copy()
    elif method_name == 'mhcnuggets':
        df = df.loc[df[method_name + "_percentile_rank_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
    elif method_name == 'acme':
        df = df.loc[df[method_name + "_score_mt"] > 0.42].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif method_name == 'mhcseqnet':
        df = df.loc[df[method_name + "_score_mt"] > 0.99].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif method_name == 'puffin':
        df = df.loc[df[method_name + "_percentile_rank_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
#     elif method_name == 'puffin':
#         df = df.loc[df[method_name + "_binding_likelihood_mt"] > 0.5].copy()
#         if advantage:
#             df = df.loc[df[method_name + "_binding_likelihood_advantage"] > 1].copy()
    elif method_name == 'mhcattnnet':
        df = df.loc[df[method_name + "_score_mt"] >= 1].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    return df


In [46]:
unsupported_alleles = {method : set() for method in all_methods}

def add_unsupported_allele(method, alleles):
    methods_set = set(all_methods)
    if (method in methods_set):
        unsupported_alleles[method].update(alleles)

for allele_name in ALLELES:
    if defineClass(allele_name) == 'II':
        add_unsupported_allele('netMHCpan-4.1', [allele_name])
        add_unsupported_allele('netMHC-3.4', [allele_name])
        add_unsupported_allele('netMHC-4.0', [allele_name])
        add_unsupported_allele('acme', [allele_name])
    else:
        add_unsupported_allele('netMHCIIpan-4.0', [allele_name])
        
    if not defineOrganism(allele_name) == 'Human':
        add_unsupported_allele('mhcseqnet', [allele_name])
        add_unsupported_allele('mhcattnnet', [allele_name])
        add_unsupported_allele('acme', [allele_name])

#special cases
add_unsupported_allele('acme',["HLA-C*04:01", "HLA-C*03:04"])
add_unsupported_allele('puffin',["HLA-C*04:01"])

In [47]:
unsupported_alleles

{'netMHC-3.4': {'H-2IEd'},
 'netMHCIIpan-4.0': {'H-2Ld'},
 'netMHC-4.0': {'H-2IEd'},
 'mhcnuggets': set(),
 'puffin': {'HLA-C*04:01'},
 'netMHCpan-4.1': {'H-2IEd'}}

In [48]:
def per_allele_filter_predictions(per_allele_df, selected_methods, advantage=False):
    per_allele_result = dict()
    for allele_name in ALLELES:
        prediction_df =  per_allele_df[allele_name]
        count = len(prediction_df)
        print(f"{allele_name}:({count}) ->", end=" ")
        for method in selected_methods:
            if allele_name not in unsupported_alleles[method]:
                prediction_df = filter_epitopes(prediction_df, method, advantage)
                count = len(prediction_df)
                print(f"{method}:({count}) ->", end=" ")
        print()
        per_allele_result[allele_name] = prediction_df
    return per_allele_result

def per_allele_add_source_info(per_allele_df, region):
    per_allele_result = dict()
    df_source =  per_region_total_df[region].copy()
    df_source = df_source.rename(columns={ "HLA Allele":"allele", "MT Epitope Seq":"peptide_mt", "WT Epitope Seq":"peptide_wt"}) # rename for convinience
    
    # No Transcript, Protein position , Transcript Expression, Transcript Support Level because it produces many duplicates
    df_source = df_source[
        [ 'allele','peptide_mt', 'peptide_wt','Chromosome', 'Start', 'Stop', 'Reference', 'Variant', 
         'Ensembl Gene ID', 
         'Variant Type','Mutation', 
         'Gene Name', 'HGVSc', 'HGVSp', 'Mutation Position', 
         'Tumor DNA Depth', 'Tumor DNA VAF', 'Tumor RNA Depth', 'Tumor RNA VAF',
         'Normal Depth', 'Normal VAF', 'Gene Expression',
         'cterm_7mer_gravy_score','max_7mer_gravy_score', 'difficult_n_terminal_residue',
       'c_terminal_cysteine', 'c_terminal_proline', 'cysteine_count','n_terminal_asparagine', 'asparagine_proline_bond_count']].drop_duplicates()
    
    
    for allele_name in ALLELES:
        df_top = per_allele_df[allele_name].copy()
        len_initial = len(df_top)
        df_merged = df_source.merge(df_top, on=["allele","peptide_mt", "peptide_wt" ])
        len_merged = len(df_merged)
        df_filtered = df_source[(df_source['peptide_mt'].isin(df_top['peptide_mt'])) & (df_source['peptide_wt'].isin(df_top['peptide_wt'])) & (df_source['allele'].isin(df_top['allele']))]
        len_filtered = len(df_filtered)
        print(f"top epitopes {len_initial}, after merging  {len_merged}, after filtering {len_filtered}")
        if not len_merged == len_initial :
            print(f"many rows for 1 epitope due to multiple Gene ID")
        
        per_allele_result[allele_name] = df_merged
    return per_allele_result

def drop_unsupported_columns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        for method in all_methods:
            if allele_name in unsupported_alleles[method]:
                columns_to_drop = [col.strip() for col in df.columns if col.startswith(method)]
            
                df = df.drop(columns=columns_to_drop, errors='ignore')    
            per_allele_df[allele_name] = df
        
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df

    
    
# def reorder_result_columns(df_filtered):  
#     df_filtered = df_filtered.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
#      df_filtered = df_filtered.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])

#     return df_filtered


def filter_and_save_to_file(per_region_per_allele_predictions, selected_methods, advantage=False):
    print("1.Filter using methods")
    per_allele_top_epitopes = dict()
    
    per_allele_top_epitopes = per_allele_filter_predictions(per_allele_predictions, selected_methods, advantage=advantage)
    print("2.Drop columns")
    per_allele_top_epitopes = drop_unsupported_columns(per_allele_top_epitopes)
    

    print("2.Write to files")
    #DATA_PATH = "/home/denklewer/Data/gbm122/" #ubuntu machine
    DATA_PATH = PREDICTION_PATH
    for allele_name, df_filtered in per_allele_top_epitopes.items():
        if advantage:
            df_filtered.to_csv("{}top_epitopes_{}_filtered+advantage_{}.csv".format(DATA_PATH, allele_name, "_".join(selected_methods)),sep=";", index=False)
        else:
            df_filtered.to_csv("{}top_epitopes_filtered_all_regions_{}.csv".format(DATA_PATH,allele_name,  "_".join(selected_methods)),sep=";", index=False)


In [49]:
per_allele_predictions[ALLELES[0]]

,allele,peptide_mt,netMHCIIpan-4.0_score_ba_mt,netMHCIIpan-4.0_score_el_mt,netMHCIIpan-4.0_affinity_mt,netMHCIIpan-4.0_percentile_rank_ba_mt,netMHCIIpan-4.0_percentile_rank_el_mt,mhcnuggets_score_mt,mhcnuggets_affinity_mt,mhcnuggets_percentile_rank_mt,...,netMHCIIpan-4.0_percentile_rank_ba_wt,netMHCIIpan-4.0_percentile_rank_el_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,mhcnuggets_percentile_rank_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt,puffin_percentile_rank_wt
0,H-2IEd,NTGRGYVLI,0.015477,0.000000,42290.62,95.00,95.00,0.097099,17486.66,39.39175,...,95.00,95.00,0.105128,16031.53,36.44675,0.018749,0.000053,0.001493,0.000000e+00,95.49525
1,H-2IEd,TGRGYVLIR,0.016791,0.000000,41693.62,95.00,95.00,0.090052,18872.05,42.27925,...,95.00,95.00,0.128964,12387.12,28.79425,0.055497,0.002107,0.004697,3.320743e-08,80.54025
2,H-2IEd,GRGYVLIRI,0.031843,0.000000,35427.52,95.00,95.00,0.156160,9229.54,22.06475,...,95.00,95.00,0.226522,4310.70,10.84550,0.116005,0.004077,0.012052,2.399052e-03,57.78325
3,H-2IEd,RGYVLIRIL,0.035624,0.000000,34007.44,95.00,95.00,0.170126,7935.15,19.13775,...,95.00,95.00,0.262568,2918.58,7.41325,0.138258,0.004254,0.016721,1.313165e-02,48.55075
4,H-2IEd,GYVLIRILR,0.028357,0.000000,36789.28,95.00,95.00,0.222630,4496.11,11.27675,...,95.00,95.00,0.428206,486.23,0.97975,0.118879,0.005188,0.011707,2.291161e-03,56.59175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,H-2IEd,RPDNTGRGYVLIRILRRAVRYSH,0.499410,0.081030,225.04,0.07,2.14,0.226715,4301.73,10.81925,...,0.02,1.80,0.242110,3641.70,9.21225,0.517064,0.014477,0.049402,6.596083e-01,0.72225
130,H-2IEd,PDNTGRGYVLIRILRRAVRYSHE,0.503210,0.076633,215.97,0.06,2.35,0.222465,4504.15,11.29675,...,0.02,2.00,0.246511,3472.34,8.78775,0.530231,0.014139,0.049178,6.814311e-01,0.58275
131,H-2IEd,GRPDNTGRGYVLIRILRRAVRYSH,0.500701,0.087037,221.92,0.07,1.88,0.207119,5317.72,13.19625,...,0.02,1.59,0.212698,5006.22,12.47475,0.486559,0.014376,0.052542,6.048162e-01,1.19925
132,H-2IEd,RPDNTGRGYVLIRILRRAVRYSHE,0.504806,0.083735,212.28,0.05,2.02,0.236684,3861.87,9.76025,...,0.02,1.72,0.252655,3249.00,8.24525,0.513499,0.014276,0.051864,6.501983e-01,0.77375


In [50]:
selected_methods = ['netMHCIIpan-4.0', 'netMHCpan-4.1','mhcnuggets','puffin']
dicc  = filter_and_save_to_file(per_allele_predictions, selected_methods, advantage=False)

1.Filter using methods
H-2IEd:(134) -> netMHCIIpan-4.0:(45) -> mhcnuggets:(32) -> puffin:(5) -> 
H-2Ld:(51) -> netMHCpan-4.1:(9) -> mhcnuggets:(5) -> puffin:(3) -> 
2.Drop columns
2.Write to files


In [51]:
selected_methods = ['netMHCIIpan-4.0', 'netMHCpan-4.1','mhcnuggets','puffin']
dicc  = filter_and_save_to_file(per_allele_predictions, selected_methods, advantage=True)

1.Filter using methods
H-2IEd:(134) -> netMHCIIpan-4.0:(32) -> mhcnuggets:(1) -> puffin:(0) -> 
H-2Ld:(51) -> netMHCpan-4.1:(9) -> mhcnuggets:(5) -> puffin:(3) -> 
2.Drop columns
2.Write to files
